# Resource estimation for the 2D Fermi-Hubbard model

This notebook estimates QPE/time-evolution resource costs for one periodic two-dimensional
Fermi-Hubbard lattice, using the plaquette Trotterization of the time evolution. It does
not include the cost of preparing a many-electron trial or ground state, so it is not a
complete ground-state-energy algorithm estimate.
See [`extended_hubbard.ipynb`](../extended_hubbard.ipynb) for more details on the model Hamiltonian.

It has two halves:

1. a **logical** resource count for the lattice size set by `EXAMPLE_SIZE`, reporting
   qubits, rotations, T gates and Toffolis in the emitted circuit; and
2. a **physical** estimate for that same lattice, which maps those logical counts onto a
   fault-tolerant architecture and reports physical qubits and runtime.

> The import cell silences QDK/Chemistry library logs so the tables stay readable. Change
> `Logger.LogLevel.off` to `Logger.LogLevel.info` and rerun that cell to see detailed
> calculation logs.


## Background: the 2D Fermi-Hubbard model

The Fermi-Hubbard Hamiltonian on a square lattice is

$$
H = -t \sum_{\langle i,j \rangle, \sigma} \left( c^\dagger_{i\sigma} c_{j\sigma} + \text{h.c.} \right)
  + U \sum_i n_{i\uparrow} n_{i\downarrow},
$$

a nearest-neighbour hopping term of amplitude $t$ and an on-site repulsion $U$. 
Each site carries two spin orbitals, so an $L \times L$ lattice needs $2L^2$ qubits under the Jordan-Wigner mapping.

We take $U = 8t$, the strong-coupling regime. The lattice is periodic in both directions.

We allow

$$
\epsilon_{\text{total}} = 0.0051 \ \text{per site},
$$

For the lattice selected by `EXAMPLE_SIZE`, this per-site budget is multiplied by $L^2$. We allocate a fraction (f=2/3) to phase estimation and the remaining (1/3) to Trotter error. QRE internally allocates an error budget of (0.01) to rotation synthesis, which should be negligible for larger (L).

In [ ]:
import math

from qdk import qsharp
from qdk_chemistry.algorithms import create
from qdk_chemistry.algorithms.state_preparation import identity_state_prep
from qdk_chemistry.data import AlgorithmRef, LatticeGeometry, QubitOperator
from qdk_chemistry.data.qubit_operator.containers.lattice import LatticeContainer
from qdk_chemistry.utils import Logger

# Silence the library logs so the tables below stay readable.
Logger.set_global_level(Logger.LogLevel.off)

#: Hopping amplitude, which sets the unit of energy.
HOPPING_T = 1.0

#: U = 8t for the strong-coupling regime.
U_OVER_T = 8.0

#: Total error budget per site, covering phase estimation, Trotter and gate synthesis.
TARGET_PRECISION_PER_SITE = 0.0051

#: Number of phase qubits in QPE.
QPE_PRECISION_BITS = 10

#: Share of the energy budget allocated to phase estimation. Minimizing the total Trotter
#: step count sum_k r_k, which scales as 1 / (f * sqrt(1 - f)) in the phase-estimation
#: share f, gives f = 2/3. Same optimum used in Campbell (arXiv:2012.09238v4, App. F).
QPE_BUDGET_FRACTION = 2.0 / 3.0

#: Plaquette Trotter order.
TROTTER_ORDER = 2

## How long each phase-estimation query evolves for

Phase estimation reads the phase $\tau E$ accumulated by evolving for a time $\tau$, so the
energy resolution is set by how long the evolution runs. We prepare the phase register in a
**sine window** rather than a uniform superposition: with $N = 2^b - 1$ queries the sine
window's phase estimate has spread

$$
\tan\!\left(\frac{\pi}{N + 2}\right),
$$

which is Heisenberg-limited and a constant factor better than the uniform window. Setting
that spread equal to the required $\epsilon_{\text{QPE}} \cdot \tau$ fixes the base evolution
time.

Bit $k$ of the phase register then evolves for `base_time` $\cdot\, 2^k$ — the `"rescale"`
power strategy — rather than repeating the evolution block $2^k$ times. Rescaling the time
inside a single block is far cheaper than repeating the block, and is exact here because the
Trotter step count is re-derived for each rescaled time.

In [ ]:
#: Lattice used for the narrative below and for the physical estimate at the end.
EXAMPLE_SIZE = 4


def evolution_schedule(size):
    """Return the energy budgets and the base evolution time for one lattice size."""
    num_sites = size * size
    energy_budget = TARGET_PRECISION_PER_SITE * num_sites
    qpe_budget = QPE_BUDGET_FRACTION * energy_budget
    trotter_budget = energy_budget - qpe_budget
    # A sine-windowed QPE phase state of N = 2^bits - 1 queries has spread
    # tan(pi / (N + 2)), which equals the required eps_QPE * tau.
    base_time = math.tan(math.pi / (2**QPE_PRECISION_BITS - 1 + 2)) / qpe_budget
    return energy_budget, qpe_budget, trotter_budget, base_time


energy_budget, qpe_budget, trotter_budget, base_time = evolution_schedule(EXAMPLE_SIZE)

## Building the time-evolution circuit

The evolution uses the **plaquette Trotterization**: the lattice is tiled twice by
four-cycles (Campbell's "pink" and "gold" tilings), each plaquette is rotated into the
momentum basis by a fermionic FFT, and the hopping term becomes diagonal there. Combined
with the on-site interaction layer, one second-order Trotter step is a symmetric product of
four layers.

The `state_prep` below is `identity_state_prep`, so the system register starts in the
vacuum/computational-zero state. That keeps the notebook scoped to the QPE/time-evolution
resource count and excludes the cost of preparing a many-electron trial or ground state.

Every layer ends up as a batch of **same-angle** $R_z$ rotations. Once a batch reaches eight
terms, it is phased through a Hamming-weight register instead of term by term: an adder tree
compresses $n$ same-angle rotations into a $\log_2 n$-bit weight, and only one rotation per
place value is applied. This is the construction of :cite:`Kan2025`, whose Methods
diagonalize every plaquette and every on-site pair into a layer of same-angle $R_z$ gates and
synthesize that layer collectively.
The adder tree is where the qubits go: a tower of $n$ same-angle rotations needs roughly $n$
scratch qubits held for as long as the weight is needed, and on an $L \times L$ lattice every
tower is $L^2$ terms long. The mapper's `max_hwp_batch_size` setting caps how many terms share
one register. Batches are phased one after another, so the scratch of one is released before
the next allocates and the peak ancilla count follows the batch rather than the whole lattice
— at the cost of one extra set of place-value rotations per batch. It is a
qubits-for-rotations knob, and which way to turn it depends on the target device: a
fault-tolerant estimate usually has the qubits to spare, while a near-term processor may not. The default is `-1`, no cap,
which is what this notebook uses.


In [ ]:
lattice = LatticeGeometry.square(
    EXAMPLE_SIZE, EXAMPLE_SIZE, periodic_x=True, periodic_y=True
)
lattice_operator = QubitOperator(container=LatticeContainer(lattice))

unitary_builder = AlgorithmRef(
    "hamiltonian_unitary_builder",
    "hubbard_plaquette",
    order=TROTTER_ORDER,
    time=base_time,
    t=HOPPING_T,
    u=U_OVER_T * HOPPING_T,
    # Bit k evolves for base_time * 2^k rather than repeating the block 2^k times.
    power_strategy="rescale",
    target_accuracy=trotter_budget,
)
circuit_builder = create(
    "qpe_circuit_builder",
    "qdk_standard",
    num_bits=QPE_PRECISION_BITS,
    unitary_builder=unitary_builder,
    controlled_circuit_mapper=AlgorithmRef(
        "controlled_circuit_mapper", "hubbard_plaquette"
    ),
)
circuit_builder.settings().set("phase_state", "sine")

state_prep = identity_state_prep(num_qubits=lattice_operator.num_qubits)
circuit = circuit_builder.run(state_prep, lattice_operator)[0]

## Counting the logical resources

`logical_counts` traces the emitted Q# program and returns the gate counts without
simulating it.

In [ ]:
qsharp_factory = circuit._qsharp_factory
if qsharp_factory is None:
    raise RuntimeError("The QPE circuit does not have Q# factory data.")
qsharp_context = getattr(qsharp_factory.program, "_qdk_context", qsharp)
counts = dict(
    qsharp_context.logical_counts(
        qsharp_factory.program,
        *qsharp_factory.parameter.values(),
    )
)
counts

## Physical resource estimation

The logical counts above say nothing about wall-clock time or physical qubit count. To get
those we map the circuit onto a concrete fault-tolerant architecture: a Majorana qubit with
a $10^{-5}$ physical error rate, error-corrected with a three-auxiliary code, running
lattice surgery, and fed by round-based magic-state factories.

In [ ]:
from qdk.qre import LatticeSurgery, PSSPC, estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

#: Majorana qubit at a 1e-5 physical error rate.
ARCHITECTURE = Majorana(error_rate=1e-5)

#: Total failure probability allowed for the whole algorithm.
MAX_ERROR = 0.01

application = circuit.get_qre_application()

trace_query = (
    application.q()
    * PSSPC.q(num_ts_per_rotation=[20, 30, 40])
    * LatticeSurgery.q(slow_down_factor=[1.0, 5.0])
)
isa_query = ThreeAux.q() * RoundBasedFactory.q(code_query=ThreeAux.q())

estimates = estimate(
    application,
    ARCHITECTURE,
    isa_query,
    trace_query,
    max_error=MAX_ERROR,
    name=f"Hubbard {EXAMPLE_SIZE}x{EXAMPLE_SIZE}",
)
estimates.add_qubit_partition_column()
estimates.add_factory_summary_column()
estimates.as_frame()

In [ ]:
plot_estimates([estimates], runtime_unit="months", figsize=(15, 7))